# Example-07: Effect of filtering on parameter uncertainties

In [1]:
# Import

import numpy
import torch

import sys
sys.path.append('..')

from harmonica.util import data_load
from harmonica.window import Window
from harmonica.data import Data
from harmonica.frequency import Frequency
from harmonica.filter import Filter

import matplotlib.pyplot as plt

torch.set_printoptions(precision=12, sci_mode=True)
print(torch.cuda.is_available())

True


In [2]:
# Set data type and device

dtype = torch.float64
device = torch.device('cpu')

In [3]:
# Estimate frequency and amplitude uncertainty from multiple realizations

# Set parameters

length = 1024

# Set signal

t = torch.linspace(1, length, length, dtype=dtype, device=device)
s = 0.10
x = 1.0*torch.cos(2.0*numpy.pi*0.12345*t) + 0.1*torch.cos(2.0*numpy.pi*2.0*0.12345*t)

# Set TbT (signal copies with different noise realizations)

w = Window.from_cosine(length, 1.0, dtype=dtype, device=device)
d = torch.stack([x + s*torch.randn(length, dtype=dtype, device=device) for _ in range(256)])
d = Data.from_data(w, d)

# Estimate frequency

d.window_remove_mean()
d.window_apply()
f = Frequency(d)
f('parabola')
m_f, s_f = f.frequency.mean().cpu().item(), f.frequency.std().cpu().item()
print(f'frequency: error={abs(0.12345 - m_f):<16.12}, spread={s_f:<16.12}')
d.reset()

# Estimate amplitude

c = 2.0/w.total*torch.sum(torch.cos(2.0*numpy.pi*m_f*t)*d.work*w.window, 1)
s = 2.0/w.total*torch.sum(torch.sin(2.0*numpy.pi*m_f*t)*d.work*w.window, 1)
a = torch.sqrt(c*c + s*s)
m_a, s_a = a.mean().cpu().item(), a.std().cpu().item()
print(f'amplitude: error={abs(1.0 - m_a):<16.12}, spread={s_a:<16.12}')
d.reset()

frequency: error=1.26410024892e-07, spread=3.81814912263e-06
amplitude: error=0.000512163724977, spread=0.0056016475787 


In [4]:
# Estimate rank/noise

l = Filter(d)
rank, noise = l.estimate_noise(limit=16, cpu=True)
print(f'min: {rank.min().cpu().item()}')
print(f'max: {rank.max().cpu().item()}')
print(f'std: {noise.mean().cpu().item()}')

min: 4
max: 4
std: 0.1026412609859475


In [5]:
# SVD

# Estimate frequency

l.filter_svd(rank=8, random=False, cpu=True)
d.window_remove_mean()
d.window_apply()
f = Frequency(d)
f('parabola')
m_f, s_f = f.frequency.mean().cpu().item(), f.frequency.std().cpu().item()
print(f'frequency: error={abs(0.12345 - m_f):<16.12}, spread={s_f:<16.12}')
d.reset()

# Estimate amplitude

l.filter_svd(rank=8, random=False, cpu=True)
c = 2.0/w.total*torch.sum(torch.cos(2.0*numpy.pi*m_f*t)*d.work*w.window, 1)
s = 2.0/w.total*torch.sum(torch.sin(2.0*numpy.pi*m_f*t)*d.work*w.window, 1)
a = torch.sqrt(c*c + s*s)
m_a, s_a = a.mean().cpu().item(), a.std().cpu().item()
print(f'amplitude: error={abs(1.0 - m_a):<16.12}, spread={s_a:<16.12}')
d.reset()

frequency: error=1.25510749571e-07, spread=8.76861530316e-07
amplitude: error=0.000498240398673, spread=0.00457096556083


In [6]:
# SVD & Hankel

# Estimate frequency

l.filter_svd(rank=8, random=False, cpu=True)
l.filter_hankel(rank=8, random=True, buffer=8, count=8, cpu=True)
d.window_remove_mean()
d.window_apply()
f = Frequency(d)
f('parabola')
m_f, s_f = f.frequency.mean().cpu().item(), f.frequency.std().cpu().item()
print(f'frequency: error={abs(0.12345 - m_f):<16.12}, spread={s_f:<16.12}')
d.reset()

# Estimate amplitude

l.filter_svd(rank=8, random=False, cpu=True)
l.filter_hankel(rank=8, random=True, buffer=8, count=8, cpu=True)
c = 2.0/w.total*torch.sum(torch.cos(2.0*numpy.pi*m_f*t)*d.work*w.window, 1)
s = 2.0/w.total*torch.sum(torch.sin(2.0*numpy.pi*m_f*t)*d.work*w.window, 1)
a = torch.sqrt(c*c + s*s)
m_a, s_a = a.mean().cpu().item(), a.std().cpu().item()
print(f'amplitude: error={abs(1.0 - m_a):<16.12}, spread={s_a:<16.12}')
d.reset()

frequency: error=8.56949437245e-08, spread=7.21431650161e-07
amplitude: error=0.000500307694795, spread=0.00452734702972


In [7]:
# RPCA

# Estimate frequency

l.filter_rpca(limit=4096, factor=1.0E-12, cpu=True)
d.window_remove_mean()
d.window_apply()
f = Frequency(d)
f('parabola')
m_f, s_f = f.frequency.mean().cpu().item(), f.frequency.std().cpu().item()
print(f'frequency: error={abs(0.12345 - m_f):<16.12}, spread={s_f:<16.12}')
d.reset()

# Estimate amplitude

l.filter_rpca(limit=4096, factor=1.E-12, cpu=True)
c = 2.0/w.total*torch.sum(torch.cos(2.0*numpy.pi*m_f*t)*d.work*w.window, 1)
s = 2.0/w.total*torch.sum(torch.sin(2.0*numpy.pi*m_f*t)*d.work*w.window, 1)
a = torch.sqrt(c*c + s*s)
m_a, s_a = a.mean().cpu().item(), a.std().cpu().item()
print(f'amplitude: error={abs(1.0 - m_a):<16.12}, spread={s_a:<16.12}')
d.reset()

frequency: error=2.58301275063e-07, spread=1.38231254689e-06
amplitude: error=0.00760342753129, spread=0.00506863357094
